In [ ]:
# SCD Phi-3.5 Mini Fine-Tuning
QLoRA fine-tuning of Phi-3.5 Mini on sickle cell disease clinical data.

- Dataset: TeslaInch/SCD-Instruction-Tuning (140 train, 11 val)
- Adapter: TeslaInch/scd-phi35-adapter
- Final val loss: 1.142
- Token accuracy: 73.0%
- Training time: 30 minutes on T4 GPU
- WandB: https://wandb.ai/tesla369-pdepth/scd-medical-llm

In [1]:
# Cell 1 after restart - reinstall
!pip install -q --upgrade transformers peft trl bitsandbytes accelerate datasets wandb


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import wandb
import os

# login to HuggingFace
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

# login to WandB
wandb.login(key=os.environ["WANDB_API_KEY"])

# confirm GPU is available
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /teamspace/studios/this_studio/.netrc
wandb: Currently logged in as: tesla369 (tesla369-pdepth) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU available: True
GPU name: Tesla T4
GPU memory: 15.6 GB


In [4]:
dataset = load_dataset("TeslaInch/SCD-Instruction-Tuning")

print(f"Dataset structure: {dataset}")
print(f"\nTrain examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"\nColumn names: {dataset['train'].column_names}")
print(f"\n--- First training example ---")
print(f"INSTRUCTION:\n{dataset['train'][0]['instruction']}")
print(f"\nINPUT:\n{dataset['train'][0]['input']}")
print(f"\nOUTPUT:\n{dataset['train'][0]['output']}")

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'source', 'category'],
        num_rows: 140
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'source', 'category'],
        num_rows: 11
    })
})

Train examples: 140
Validation examples: 11

Column names: ['instruction', 'input', 'output', 'source', 'category']

--- First training example ---
INSTRUCTION:
You are a medical AI assistant specialised in sickle cell disease. Answer the following clinical question accurately and completely.

INPUT:
Clinical case:
A 3-year-old boy in Lagos is brought to a general practitioner with his third episode of painful swollen hands in 6 months, recurrent fevers, and pallor. His mother says a previous doctor told her he has 'the blood problem.' No formal diagnosis has been made. No newborn screening was done.

Question: What is the likely diagnosis, what confirmatory test is needed, and what immediate care 

In [22]:
def format_example(example):
    """
    Phi-3 expects this exact chat format.
    <|user|> is the human turn
    <|assistant|> is the model turn
    <|end|> closes each turn
    """
    text = f"<|user|>\n{example['instruction']}\n{example['input']}<|end|>\n<|assistant|>\n{example['output']}<|end|>"
    return {"text": text}

# apply formatting to both splits
train_formatted = dataset['train'].map(format_example)
val_formatted = dataset['validation'].map(format_example)

# verify it looks right
print("--- Formatted example ---")
print(train_formatted[0]['text'])

--- Formatted example ---
<|user|>
You are a medical AI assistant specialised in sickle cell disease. Answer the following clinical question accurately and completely.
Clinical case:
A 3-year-old boy in Lagos is brought to a general practitioner with his third episode of painful swollen hands in 6 months, recurrent fevers, and pallor. His mother says a previous doctor told her he has 'the blood problem.' No formal diagnosis has been made. No newborn screening was done.

Question: What is the likely diagnosis, what confirmatory test is needed, and what immediate care must be initiated while awaiting results?<|end|>
<|assistant|>
Likely HbSS sickle cell disease — dactylitis, recurrent fevers, and pallor in a Nigerian child is SCD until proven otherwise. Confirmatory test: HPLC (gold standard) — send urgently. While awaiting results: 1) Start prophylactic penicillin V immediately — do not wait for confirmation given high clinical suspicion. 2) Start folic acid supplementation. 3) Counsel 

In [16]:
from transformers import BitsAndBytesConfig
import torch

# Cell 5 - use Phi-3.5 instead
MODEL_ID = "microsoft/Phi-3.5-mini-instruct"  # newer, no RoPE bug

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Phi-3.5 Mini in 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    attn_implementation="eager",
)

print(f"Model loaded successfully")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading Phi-3.5 Mini in 4-bit quantization...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Model loaded successfully
GPU memory used: 4.95 GB


In [17]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# prepare model for training
model = prepare_model_for_kbit_training(model)

# configure LoRA
lora_config = LoraConfig(
    r=16,                    # rank — size of adapter matrices
    lora_alpha=32,           # scaling factor — 2x rank is standard
    target_modules=[         # which layers to inject adapters into
        "q_proj",
        "k_proj", 
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,       # small dropout prevents overfitting
    bias="none",             # don't train bias terms
    task_type="CAUSAL_LM",   # language modelling task
)

# inject adapters into model
model = get_peft_model(model, lora_config)

# show how many parameters are trainable
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


In [18]:
import wandb

# close any existing wandb run first
wandb.finish()

# start fresh run
wandb.init(
    project="scd-medical-llm",
    name="phi35-mini-scd-v1-bf16",
    config={
        "model": "Phi-3.5-mini-instruct",
        "dataset": "TeslaInch/SCD-Instruction-Tuning",
        "train_examples": len(train_formatted),
        "val_examples": len(val_formatted),
        "lora_rank": 16,
        "lora_alpha": 32,
        "precision": "bfloat16",
    }
)

from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./scd-phi35-adapter",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    processing_class=tokenizer,
)

print("Trainer ready — starting training...")
trainer.train()

Trainer ready — starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,1.368606,1.350721,1.359826,36779.000000,0.692461
50,1.113764,1.172675,1.132857,75123.000000,0.724235
75,1.041640,1.143675,1.061770,111444.000000,0.726614
90,1.006548,1.142053,1.062278,134250.000000,0.729921


TrainOutput(global_step=90, training_loss=1.26291610399882, metrics={'train_runtime': 1843.845, 'train_samples_per_second': 0.38, 'train_steps_per_second': 0.049, 'total_flos': 3856885083353088.0, 'train_loss': 1.26291610399882, 'epoch': 5.0})

In [19]:
# Cell 9 — save adapter
adapter_path = "./scd-phi35-adapter-final"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

Adapter saved to ./scd-phi35-adapter-final


In [20]:
# Cell 10 — push to HuggingFace
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(
    repo_id="TeslaInch/scd-phi35-adapter",
    repo_type="model",
    exist_ok=True
)

trainer.model.push_to_hub("TeslaInch/scd-phi35-adapter")
tokenizer.push_to_hub("TeslaInch/scd-phi35-adapter")
print("Adapter pushed to HuggingFace")
wandb.finish()

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Adapter pushed to HuggingFace


eval/entropy,█▃▁▁
eval/loss,█▂▁▁
eval/mean_token_accuracy,▁▇▇█
eval/num_tokens,▁▄▆█
eval/runtime,▆▁▅█
eval/samples_per_second,▁██▁
eval/steps_per_second,███▁
train/entropy,▅▄▅█▇▅▄▃▄▃▂▂▂▁▂▂▁▂
train/epoch,▁▁▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇████
train/global_step,▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇████
+5,...


In [21]:
# Cell 11 — test your fine-tuned model
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=300,
)

test_question = "A 4-year-old Nigerian child presents with painful swollen hands and fever. What is the diagnosis and immediate management?"

prompt = f"<|user|>\nYou are a medical AI assistant specialised in sickle cell disease. Answer the following clinical question accurately and completely.\n{test_question}<|end|>\n<|assistant|>\n"

result = pipe(prompt, do_sample=False)
print(result[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it str

<|user|>
You are a medical AI assistant specialised in sickle cell disease. Answer the following clinical question accurately and completely.
A 4-year-old Nigerian child presents with painful swollen hands and fever. What is the diagnosis and immediate management?<|end|>
<|assistant|>
Sickle cell crisis with hand-foot syndrome. Immediate management: IV hydration, analgesia (paracetamol, morphine), and exchange transfusion if severe.
